# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a step-by-step template for loading and exploring the FAIR^2 dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is described using a [Croissant](https://mlcommons.org/croissant/) schema available at:

```
https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json
```


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant


## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata and package
dataset = mlc.Dataset(croissant_url)

# Print high-level metadata summary
print('Dataset Name: ', dataset.metadata.name)
print('\nDescription: ' + dataset.metadata.description)
print('\nPublished:', getattr(dataset.metadata, 'datePublished', 'N/A'))
print('\nKeywords:', getattr(dataset.metadata, 'keywords', []))
print('\nLicense:', getattr(dataset.metadata, 'license', 'N/A'))
print('\nCoverage:', getattr(dataset.metadata, 'spatialCoverage', 'N/A'), '|', getattr(dataset.metadata, 'temporalCoverage', 'N/A'))


## 2. Data Overview
Review available record sets and fields using their `@id` values.

We use `dataset.record_sets` and inspect each `record_set` and its `fields`, referencing everything by `@id` as recommended for Croissant datasets.

In [ ]:
# List all available record sets with their @id and field @ids
print("Available Record Sets:")
for record_set in dataset.record_sets:
    print(f"- RecordSet @id: {record_set.id}\n  Name: {record_set.name}")
    print("  Fields:")
    for field in getattr(record_set, 'fields', []):
        print(f"    - Field @id: {field.id}   Name: {field.name}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

Here, we will load all record sets, referencing them by their `@id`, and create a Pandas DataFrame for each. To find the right `@id`s, refer to the previous overview cell's output.

In [ ]:
# Collect all record set @ids
record_sets_ids = [rs.id for rs in dataset.record_sets]
print("Record set @ids found:")
print(record_sets_ids)

# Create a dictionary of dataframes per record set
dataframes = {}
for record_set_id in record_sets_ids:
    print(f"\nLoading data for RecordSet with @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    if not df.empty:
        print(f"Columns: {df.columns.tolist()}")
        display(df.head())
    else:
        print("No records found.")


## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering, normalizing numeric fields, and grouping by categorical fields.

Below, we select a record set and numeric field by their `@id` as shown by the previous outputs.

_**Note:** You may need to adapt `record_set_id` and `numeric_field_id` if you use a different record set or field._

In [ ]:
# Example EDA on one record set
# --- Please replace these with @ids present in the overview output above ---
if len(dataframes) > 0:
    # Pick first record set with data
    sample_record_set_id = next(k for k, v in dataframes.items() if not v.empty)
    df = dataframes[sample_record_set_id]
    print(f"\nUsing RecordSet: {sample_record_set_id}")

    numeric_fields = df.select_dtypes(include=['number']).columns.tolist()
    print(f"Numeric fields in this set: {numeric_fields}")

    if numeric_fields:
        numeric_field = numeric_fields[0]

        # Threshold filter example
        threshold = df[numeric_field].mean() if not df[numeric_field].isnull().all() else 0
        filtered_df = df[df[numeric_field] > threshold]
        print(f"\nFiltered records where {numeric_field} > {threshold}:")
        display(filtered_df.head())

        # Normalization example
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Grouping by a categorical field if one exists
        cat_fields = df.select_dtypes(include=['object', 'category']).columns.tolist()
        group_field = None
        for col in cat_fields:
            if len(df[col].unique()) > 1 and len(df[col].unique()) < len(df) / 2:
                group_field = col
                break
        
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(f"\nGrouped average of {numeric_field} by {group_field}:")
            print(grouped_df.head())
    else:
        print("No numeric fields detected in this record set.")
else:
    print("No record sets with data found.")

## 5. Visualization
Visualize data distributions and relationships between fields in the dataset.

Below is a basic histogram and boxplot using the sample record set and numeric field chosen above.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize, if we have numeric data
if 'numeric_field' in locals() and not df.empty:
    plt.figure(figsize=(12,4))
    plt.subplot(1,2,1)
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f'Histogram of {numeric_field}')

    plt.subplot(1,2,2)
    sns.boxplot(y=df[numeric_field])
    plt.title(f'Boxplot of {numeric_field}')

    plt.tight_layout()
    plt.show()

    if 'group_field' in locals() and group_field is not None:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f'{numeric_field} by {group_field}')
        plt.xticks(rotation=30)
        plt.show()
else:
    print('No numeric or group field detected for visualization.')

## 6. Conclusion

In this notebook, we used the `mlcroissant` library to:
- Load the FAIR^2 dataset from its Croissant schema using its URL
- Enumerate and inspect available record sets and fields by their `@id`
- Extract tabular data into Pandas DataFrames for each record set
- Perform basic exploratory data analysis, including filtering, normalization, and grouping using Croissant `@id`s
- Visualize data distributions and relationships

For further analysis and modeling, refer to the Croissant documentation, and use field `@id`s for robust, schema-compliant data pipelines.

_If you wish to adapt this notebook to a different dataset, update the `croissant_url` and re-run the overview to adjust field and record set references as needed!_